In [11]:
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder

df = pl.DataFrame({
    "age": [25, 30, None, 22, 150, 28, 35],
    "salary": [50000, 60000, 55000, None, 70000, 52000, 120000],
    "city": ["Delhi", "Mumbai", "Delhi", None, "Mumbai", "Bangalore", "Delhi"],
    "gender": ["Male", "Female", "Female", "Male", "Male", "Female", "Male"]
})

print(df.null_count())
print(df.describe())

df = df.with_columns([
    pl.col("age").fill_null(pl.col("age").median()),
    pl.col("salary").fill_null(pl.col("salary").median()),
    pl.col("city").fill_null("Missing")
])

def cap_outliers(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return series.clip(lower, upper)

df = df.with_columns([
    cap_outliers(pl.col("age")).alias("age")
])

le = LabelEncoder()
df = df.with_columns(
    pl.Series("gender_encoded", le.fit_transform(df["gender"].to_numpy()))
)
print(df)

shape: (1, 4)
┌─────┬────────┬──────┬────────┐
│ age ┆ salary ┆ city ┆ gender │
│ --- ┆ ---    ┆ ---  ┆ ---    │
│ u32 ┆ u32    ┆ u32  ┆ u32    │
╞═════╪════════╪══════╪════════╡
│ 1   ┆ 1      ┆ 1    ┆ 0      │
└─────┴────────┴──────┴────────┘
shape: (9, 5)
┌────────────┬───────────┬──────────────┬───────────┬────────┐
│ statistic  ┆ age       ┆ salary       ┆ city      ┆ gender │
│ ---        ┆ ---       ┆ ---          ┆ ---       ┆ ---    │
│ str        ┆ f64       ┆ f64          ┆ str       ┆ str    │
╞════════════╪═══════════╪══════════════╪═══════════╪════════╡
│ count      ┆ 6.0       ┆ 6.0          ┆ 6         ┆ 7      │
│ null_count ┆ 1.0       ┆ 1.0          ┆ 1         ┆ 0      │
│ mean       ┆ 48.333333 ┆ 67833.333333 ┆ null      ┆ null   │
│ std        ┆ 50.002667 ┆ 26536.13888  ┆ null      ┆ null   │
│ min        ┆ 22.0      ┆ 50000.0      ┆ Bangalore ┆ Female │
│ 25%        ┆ 25.0      ┆ 52000.0      ┆ null      ┆ null   │
│ 50%        ┆ 30.0      ┆ 60000.0      ┆ null  